# Role-Aware SAAMR Quickstart

**Author:** Joseph R. Laforet Jr.

This notebook demonstrates the expected workflow for building role-aware MuPT SAAMR systems from SMILES strings and exercising the new RDKit role-aware export/import path.

The target hierarchy is:

```text
UNIVERSE -> SEGMENT -> RESIDUE -> PARTICLE
```

This notebook is intentionally self-contained. The current MuPT PR does not include the copolymer builder routine, so a small role-aware builder is defined inline. We are considering abstracting these setup utilities into a shared examples module once the API settles.

Generated SDF files are written under `examples_system/role_aware_saamr_outputs/` and are ignored by git.

## 1. Environment Check

The notebooks in this branch expect MuPT to be installed from, or importable from, the local `mupt/` checkout on branch `issue-51/refactor-rdkit-exporter`.

In [ ]:
from pathlib import Path
import sys


def find_examples_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "examples_system").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate the mupt-examples repository root")


EXAMPLES_ROOT = find_examples_root()
LOCAL_MUPT_SOURCE = EXAMPLES_ROOT / "mupt"
if LOCAL_MUPT_SOURCE.exists():
    sys.path.insert(0, str(LOCAL_MUPT_SOURCE))

import mupt

print(f"mupt import path: {mupt.__file__}")
print(f"examples root: {EXAMPLES_ROOT}")

## 2. Imports

In [ ]:
from collections import Counter
from dataclasses import dataclass

import networkx as nx
import numpy as np
from rdkit import Chem

from mupt.builders.random_walk import AngleConstrainedRandomWalk
from mupt.geometry.coordinates.directions import random_unit_vector
from mupt.interfaces.smiles import primitive_from_smiles
from mupt.interfaces.rdkit import primitive_from_rdkit, primitive_to_rdkit_mols
from mupt.mupr.primitives import Primitive
from mupt.mupr.topology import TopologicalStructure
from mupt.roles import PrimitiveRole

## 3. Define Repeat-Unit Chemistry

The example uses a small PSU/PES-like random copolymer. The `*` atoms are linker placeholders. Atom map numbers `[...:1]` and `[...:2]` mark polymer traversal direction.

In [ ]:
rep_unit_smiles: dict[str, str] = {
    "head": "[H]-[O:1]c1ccc(cc1)S(=O)(=O)c1cc[c:2](cc1)-*",
    "bisphenol_S": "*-[O:1]c1ccc(cc1)S(=O)(=O)c1cc[c:2](cc1)-*",
    "bisphenol_A": "*-[O:1]c1ccc(cc1)C(-C)(-C)c1cc[c:2](cc1)-*",
    "tail": "*-[O:1]c1ccc(cc1)S(=O)(=O)c1ccc(cc1)[O:2]-[H]",
}

mid_distribution: dict[str, float] = {
    "bisphenol_S": 0.4,
    "bisphenol_A": 0.6,
}

resname_map: dict[str, str] = {
    "head": "HEA",
    "bisphenol_S": "BPS",
    "bisphenol_A": "BPA",
    "tail": "TYL",
}

n_chains = 3
chain_len_min = 3
chain_len_max = 6
random_seed = 42

# Simple placement parameters for this tutorial. These are not a packing
# algorithm; they just prevent the independently embedded repeat units and
# chains from starting on top of one another.
inter_residue_bond_length = 1.5
angle_max_rad = np.pi / 4
chain_spacing = 8.0

print(f"Building {n_chains} chains with lengths in [{chain_len_min}, {chain_len_max}]")

## 4. Local Role-Aware Builder

This temporary builder mirrors the role protocol expected by the new RDKit exporter. It is deliberately small and notebook-local so the examples can exercise the PR before a public copolymer builder API exists.

In [ ]:
@dataclass(frozen=True)
class BuiltSystem:
    primitive: Primitive
    chain_sequences: list[list[str]]


def sequence_repeat_units(
    chain_len: int,
    mid_distrib: dict[str, float],
    rng: np.random.Generator,
) -> list[str]:
    """Return one head-to-tail repeat-unit sequence."""
    if chain_len < 2:
        raise ValueError("chain_len must be at least 2 to include head and tail units")

    mid_names = list(mid_distrib)
    probabilities = np.array([mid_distrib[name] for name in mid_names], dtype=float)
    probabilities = probabilities / probabilities.sum()
    middle = rng.choice(mid_names, size=chain_len - 2, p=probabilities).astype(object)
    return ["head", *map(str, middle), "tail"]


def build_role_aware_lexicon(rep_unit_smiles: dict[str, str]) -> dict[str, Primitive]:
    """Build RESIDUE -> PARTICLE repeat units from SMILES strings."""
    lexicon: dict[str, Primitive] = {}
    for name, smiles in rep_unit_smiles.items():
        residue = primitive_from_smiles(
            smiles,
            ensure_explicit_Hs=True,
            embed_positions=True,
            label=name,
        )
        residue.role = PrimitiveRole.RESIDUE
        for atom in residue.children:
            atom.role = PrimitiveRole.PARTICLE
        lexicon[name] = residue
    return lexicon


def build_role_aware_copolymer_system(
    rep_unit_smiles: dict[str, str],
    mid_distrib: dict[str, float],
    n_chains: int,
    chain_len_min: int,
    chain_len_max: int,
    random_seed: int = 42,
    inter_residue_bond_length: float = 1.5,
    angle_max_rad: float = np.pi / 4,
    chain_spacing: float = 8.0,
) -> BuiltSystem:
    """Build a small role-aware SAAMR copolymer system."""
    rng = np.random.default_rng(random_seed)
    lexicon = build_role_aware_lexicon(rep_unit_smiles)
    universe = Primitive(label="psu_pes_demo", role=PrimitiveRole.UNIVERSE)
    chain_sequences: list[list[str]] = []

    chain_lengths = rng.integers(chain_len_min, chain_len_max + 1, size=n_chains)
    for chain_idx, chain_len in enumerate(chain_lengths):
        segment = Primitive(label=f"chain_{chain_idx:03d}", role=PrimitiveRole.SEGMENT)
        sequence = sequence_repeat_units(int(chain_len), mid_distrib, rng)
        chain_sequences.append(sequence)

        residue_handles = []
        for residue_name in sequence:
            residue = lexicon[residue_name].copy()
            residue.role = PrimitiveRole.RESIDUE
            for atom in residue.children:
                atom.role = PrimitiveRole.PARTICLE
            residue_handles.append(segment.attach_child(residue))

        # set_topology consumes the linker connectors generated from SMILES and
        # pairs neighboring repeat units along the chain. Reviewers should note
        # this is the piece we want to abstract into a reusable setup module.
        segment.set_topology(
            nx.path_graph(residue_handles, create_using=TopologicalStructure),
            max_registration_iter=100,
        )

        # SMILES embedding gives each repeat unit a local conformation, but it
        # does not place repeat units or chains relative to one another. Without
        # this placement step, atoms from different residues/chains can overlap,
        # which produces infinite forces and NaN coordinates during OpenMM
        # minimization. This simple random-walk placement is intentionally modest:
        # it prevents obvious overlaps but is not a full melt-packing algorithm.
        placement = AngleConstrainedRandomWalk(
            bond_length=inter_residue_bond_length,
            angle_max_rad=angle_max_rad,
            initial_point=np.array([chain_idx * chain_spacing, 0.0, 0.0]),
            initial_direction=random_unit_vector(),
        )
        for handle, transform in placement.generate_placements(segment):
            segment.children_by_handle[handle].rigidly_transform(transform)

        universe.attach_child(segment)

    return BuiltSystem(primitive=universe, chain_sequences=chain_sequences)

## 5. Build and Validate a SAAMR System

In [ ]:
built = build_role_aware_copolymer_system(
    rep_unit_smiles=rep_unit_smiles,
    mid_distrib=mid_distribution,
    n_chains=n_chains,
    chain_len_min=chain_len_min,
    chain_len_max=chain_len_max,
    random_seed=random_seed,
    inter_residue_bond_length=inter_residue_bond_length,
    angle_max_rad=angle_max_rad,
    chain_spacing=chain_spacing,
)
univprim = built.primitive

print(univprim.hierarchy_summary(to_depth=2))
print("Sequences:")
for idx, sequence in enumerate(built.chain_sequences):
    print(f"  chain_{idx:03d}: {' - '.join(sequence)}")

In [ ]:
assert univprim.role == PrimitiveRole.UNIVERSE
assert all(segment.role == PrimitiveRole.SEGMENT for segment in univprim.children)
assert all(residue.role == PrimitiveRole.RESIDUE for segment in univprim.children for residue in segment.children)
assert all(atom.role == PrimitiveRole.PARTICLE for atom in univprim.leaves)

n_residues = sum(len(segment.children) for segment in univprim.children)
print(f"Segments: {len(univprim.children)}")
print(f"Residues: {n_residues}")
print(f"Particles: {len(univprim.leaves)}")

In [ ]:
def minimum_pair_distance(positions: np.ndarray) -> float:
    """Return the nearest-neighbor distance without an O(N^2) matrix."""
    from scipy.spatial import cKDTree

    distances, _ = cKDTree(positions).query(positions, k=2)
    return float(np.min(distances[:, 1]))


all_positions = np.vstack([atom.shape.centroid for atom in univprim.leaves if atom.shape is not None])
min_distance = minimum_pair_distance(all_positions)
print(f"Minimum all-atom distance before RDKit export: {min_distance:.3f} Å")
assert min_distance > 0.05, "Detected overlapping atom coordinates before export"


## 6. Export One RDKit Molecule Per Segment

`primitive_to_rdkit_mols()` expects the role-aware hierarchy and returns one RDKit `Mol` per SEGMENT. The RDKit output includes PDB-compatible residue metadata plus MuPT-specific atom props that preserve segment/residue/particle labels.

In [ ]:
rdkit_mols = primitive_to_rdkit_mols(univprim, resname_map=resname_map)

print(f"Exported {len(rdkit_mols)} RDKit molecules")
for idx, mol in enumerate(rdkit_mols):
    Chem.SanitizeMol(Chem.Mol(mol))
    atom0 = mol.GetAtomWithIdx(0)
    pdb_info = atom0.GetPDBResidueInfo()
    wildcard_count = sum(atom.GetAtomicNum() == 0 for atom in mol.GetAtoms())
    print(
        f"  mol {idx}: atoms={mol.GetNumAtoms()}, bonds={mol.GetNumBonds()}, "
        f"wildcards={wildcard_count}, chain={pdb_info.GetChainId()}, "
        f"first_resid={pdb_info.GetResidueNumber()}, "
        f"mupt_segment_index={atom0.GetIntProp('mupt_segment_index')}"
    )

## 7. Inspect PDB-Compatible Indexing

PDB chain IDs are one-character fields. Instead of assigning every disconnected MuPT segment a unique chain letter, MuPT numbers residues globally and overflows to the next chain ID every 9999 residues. The authoritative segment identity remains in `mupt_segment_index` and `mupt_segment_label`.

In [ ]:
for idx, mol in enumerate(rdkit_mols):
    residues = []
    seen = set()
    for atom in mol.GetAtoms():
        info = atom.GetPDBResidueInfo()
        key = (info.GetChainId(), info.GetResidueNumber(), info.GetResidueName().strip())
        if key not in seen:
            residues.append(key)
            seen.add(key)
    print(f"mol {idx}: first residues {residues[:5]}")

## 8. Write SDF Files for Downstream Notebooks

These files are intentionally ignored by git. They are consumed by `Role_Aware_SAAMR_OpenFF_OpenMM.ipynb`. RDKit SDF files do not automatically serialize per-atom Python properties, so this cell explicitly writes MuPT atom metadata as RDKit atom-property lists before saving.


In [ ]:
output_dir = EXAMPLES_ROOT / "examples_system" / "role_aware_saamr_outputs" / "sdf"
output_dir.mkdir(parents=True, exist_ok=True)

MUPT_ATOM_PROPS_FOR_SDF = [
    # RDKit SDF reload does not preserve AtomPDBResidueInfo directly, so write
    # both PDB-style atom metadata and MuPT hierarchy metadata as atom-property
    # lists for downstream OpenFF/OpenMM notebooks.
    "chain_id",
    "residue_id",
    "residue_name",
    "mupt_segment_index",
    "mupt_segment_label",
    "mupt_residue_index",
    "mupt_residue_label",
    "mupt_particle_index",
    "mupt_particle_label",
]


def prepare_mupt_sdf_atom_props(mol: Chem.Mol) -> None:
    """Store MuPT atom props as SDF atom-property lists before writing."""
    for prop_name in MUPT_ATOM_PROPS_FOR_SDF:
        Chem.CreateAtomStringPropertyList(mol, prop_name)


sdf_paths = []
for idx, mol in enumerate(rdkit_mols):
    prepare_mupt_sdf_atom_props(mol)
    path = output_dir / f"psu_pes_chain_{idx:03d}.sdf"
    writer = Chem.SDWriter(str(path))
    writer.write(mol)
    writer.close()
    sdf_paths.append(path)

print("Wrote SDF files:")
for path in sdf_paths:
    print(f"  {path.relative_to(EXAMPLES_ROOT)}")


## 9. SDF Reload and SAAMR Round Trip

In [ ]:
supplier = Chem.SDMolSupplier(str(sdf_paths[0]), removeHs=False, sanitize=False)
loaded_mol = supplier[0]
assert loaded_mol is not None

round_trip = primitive_from_rdkit(loaded_mol, denest=False)
segment = round_trip.children[0]

print(round_trip.hierarchy_summary(to_depth=3))
assert round_trip.role == PrimitiveRole.UNIVERSE
assert segment.role == PrimitiveRole.SEGMENT
assert all(residue.role == PrimitiveRole.RESIDUE for residue in segment.children)
assert all(atom.role == PrimitiveRole.PARTICLE for atom in round_trip.leaves)
assert [residue.label for residue in segment.children] == built.chain_sequences[0]
print(f"Round-trip residues: {[residue.label for residue in segment.children]}")


## 10. What Comes Next

The companion notebook `Role_Aware_SAAMR_OpenFF_OpenMM.ipynb` starts from the SDF files written here and demonstrates the downstream handoff to OpenFF/OpenMM, with optional template cells for GROMACS and LAMMPS export.